## 5.1 Simple RNN前向传播 - 前向传播过程

#### 1. 这一节我们要解决什么问题 🎯

前面我们已经分别学习了两部分结构：

* 第一部分是 RNN层。  
  我们知道，RNN层最核心的任务不是直接给出最终结果，而是：  
  根据当前输入 $x_t$ 和上一时刻隐藏状态 $h_{t-1}$，计算当前隐藏状态 $h_t$。
* 第二部分是 全连接层。  
  我们知道，全连接层的任务是：  
  把 RNN层产生的隐藏状态 $h_t$，进一步映射成最终任务输出。

所以到了这一节，我们就要把这两部分真正串起来，理解完整的前向传播流程。

这一节主要解决下面几个问题：

* 什么叫 Simple RNN 的前向传播？
* 在一个时间步里，RNN层和全连接层分别做什么？
* 一整个序列输入进来后，模型是怎么一步一步往前计算的？
* 隐藏状态在前向传播中到底起什么作用？
* 最终输出又是在什么位置产生的？

> Simple RNN 的前向传播，就是在时间维度上不断重复执行：先由 RNN层更新隐藏状态，再由全连接层根据隐藏状态产生输出。


#### 2. 从整体上先看：前向传播到底在传播什么 🔄

##### 2.1 前向传播的本质

所谓前向传播（Forward Propagation），本质上就是：

> 给定输入，按照网络的计算规则，从前往后一步一步算出中间结果和最终输出。

在我们以前学习的 MLP 中，前向传播通常是：

输入层 $\rightarrow$ 隐藏层 $\rightarrow$ 输出层

也就是沿着“层”的方向，从左到右一层层计算。

##### 2.2 RNN 的前向传播为什么不一样？

RNN 和普通前馈网络最大的不同在于：

> RNN 不只是沿“层”传播，还会沿“时间”传播。

也就是说，RNN 的前向传播有两条主线：

* 第一条：单个时间步内部的层间传播  
  在一个时间步内，数据会经历：  
  输入 $\rightarrow$ RNN层 $\rightarrow$ 隐藏状态 $\rightarrow$ 全连接层 $\rightarrow$ 输出
* 第二条：多个时间步之间的时间传播  
  在整个序列中，隐藏状态会经历：  
  $h_0 \rightarrow h_1 \rightarrow h_2 \rightarrow h_3 \rightarrow \dots$

所以你可以把 RNN 的前向传播理解成：

> 在时间维度上不断重复同一个“RNN层 + 全连接层”的计算结构。

#### 3. 先看单个时间步的前向传播 🧩

为了先把问题简化，我们先只看一个时间步 $t$。

在第 $t$ 个时间步，RNN 会接收两部分信息：

* 当前输入 $x_t$
* 上一时刻隐藏状态 $h_{t-1}$

然后按下面这个公式计算当前隐藏状态：

$h_t = f(W_x x_t + W_h h_{t-1} + b)$

如果当前时间步需要输出，那么再进一步计算：

$o_t = W_y h_t + b_y$

##### 3.1 第一步：RNN层先计算当前隐藏状态

RNN层会接收两部分输入：

* 当前输入 $x_t$
* 上一时刻隐藏状态 $h_{t-1}$

然后根据公式计算当前隐藏状态：

$h_t = f(W_x x_t + W_h h_{t-1} + b)$

如果激活函数采用 $tanh$，则也可以写成：

$h_t = tanh(W_x x_t + W_h h_{t-1} + b)$

这里的含义是：

* $W_x x_t$：当前输入带来的信息
* $W_h h_{t-1}$：上一时刻记忆带来的信息
* $b$：偏置项
* $f$：非线性激活函数

所以这一步本质上就是：

> RNN层把“当前输入”和“历史记忆”融合起来，得到当前时刻新的隐藏状态。

##### 3.2 第二步：全连接层根据隐藏状态生成输出

RNN层算出隐藏状态之后，如果当前时间步需要输出，那么再把 $h_t$ 输入到全连接层：

$o_t = W_y h_t + b_y$

然后根据任务需要，再经过输出激活函数：

$y_t = g(o_t)$

这里：

* $h_t$：RNN层输出的隐藏状态
* $W_y$：全连接层权重
* $b_y$：全连接层偏置
* $o_t$：线性输出
* $g$：任务对应的激活函数

例如：

* 多分类时可能是 $softmax$
* 二分类时可能是 $sigmoid$
* 回归时可能直接线性输出

所以在一个时间步中，完整流程应该理解成：

> 当前输入 + 上一时刻记忆 $\rightarrow$ RNN层更新隐藏状态 $\rightarrow$ 全连接层生成当前输出

也就是：

$x_t + h_{t-1} \rightarrow h_t \rightarrow o_t \rightarrow y_t$

#### 4. 单个时间步的计算流程

##### 4.1 第一步：接收当前输入 $x_t$

这表示当前时刻新到来的数据。

例如：

* 在文本中，可能是当前单词的词向量
* 在时间序列中，可能是当前时刻的多个数值特征

##### 4.2 第二步：接收上一时刻隐藏状态 $h_{t-1}$

这表示到上一时刻为止，模型已经保留下来的历史信息。

所以 RNN 在当前时刻不是“只看当前输入”，而是：

> 当前输入 + 历史记忆，一起参与当前计算

##### 4.3 第三步：RNN层先做线性组合

RNN层先分别对这两部分做线性变换：

* 当前输入部分：$W_x x_t$
* 历史状态部分：$W_h h_{t-1}$

然后把它们相加，再加上偏置：

$a_t = W_x x_t + W_h h_{t-1} + b$

这一项 $a_t$ 表示：

> 当前信息和过去信息融合后的线性结果。

##### 4.4 第四步：RNN层经过激活函数，得到当前隐藏状态 $h_t$

然后经过非线性激活函数：

$h_t = f(a_t)$

如果使用 $tanh$，则：

$h_t = tanh(a_t)$

所以隐藏状态 $h_t$ 的本质就是：

> 当前时刻对“当前输入 + 过去记忆”的综合表示。

这一步结束后，RNN层的核心工作就完成了。

##### 4.5 第五步：全连接层根据 $h_t$ 生成输出

如果当前时间步需要输出，那么继续把隐藏状态送入全连接层：

$o_t = W_y h_t + b_y$

然后根据任务需要得到最终输出：

$y_t = g(o_t)$

所以要注意：

> 最终输出不是 RNN层直接算出来的，而是隐藏状态再经过全连接层映射得到的。

#### 5. 一整个序列的前向传播过程 ⏱️

真正的 RNN 不是只算一个时间步，而是要处理一个完整序列。

假设输入序列为：

$x_1, x_2, x_3, \dots, x_T$

那么 RNN 的前向传播就是沿着时间步不断重复刚才那套流程。

##### 5.1 第 1 个时间步

先给定初始隐藏状态 $h_0$

在基础阶段，我们通常默认：

$h_0 = 0$

然后第 $1$ 步，RNN层计算：

$h_1 = f(W_x x_1 + W_h h_0 + b)$

如果这一时刻需要输出，再由全连接层计算：

$o_1 = W_y h_1 + b_y$

$y_1 = g(o_1)$

##### 5.2 第 2 个时间步

现在第 $1$ 步已经产生了隐藏状态 $h_1$，它会作为第 $2$ 步的历史记忆继续传下去。

所以第 $2$ 步，RNN层计算：

$h_2 = f(W_x x_2 + W_h h_1 + b)$

如果需要输出，则：

$o_2 = W_y h_2 + b_y$

$y_2 = g(o_2)$

##### 5.3 第 3 个时间步

同理继续：

$h_3 = f(W_x x_3 + W_h h_2 + b)$

如果需要输出，则：

$o_3 = W_y h_3 + b_y$

$y_3 = g(o_3)$

##### 5.4 一般化写法

对于任意时间步 $t$，都满足：

$h_t = f(W_x x_t + W_h h_{t-1} + b)$

如果当前时刻需要输出，则：

$o_t = W_y h_t + b_y$

$y_t = g(o_t)$

所以一整个序列的前向传播，本质上就是：

> 在每一个时间步，先由 RNN层更新隐藏状态，再由全连接层根据隐藏状态产生输出。

#### 6. 从“RNN层 + 全连接层”的角度看完整序列流程

现在我们把整个序列过程再从层的角度总结一遍，会更清楚。

##### 6.1 RNN层在整个序列中做什么？

RNN层在每一个时间步重复执行同一件事：

> 用当前输入和上一时刻隐藏状态，计算当前隐藏状态。

所以 RNN层真正负责的是：

* 隐藏状态的生成
* 历史信息的传递
* 时序特征的提取

它在整个序列中的主线是：

$h_0 \rightarrow h_1 \rightarrow h_2 \rightarrow \dots \rightarrow h_T$

##### 6.2 全连接层在整个序列中做什么？

全连接层的任务则是：

> 根据当前时间步的隐藏状态，把内部特征映射成任务输出。

所以如果每一步都需要输出，那么它会在每一步都执行：

$h_t \rightarrow o_t \rightarrow y_t$

如果只在最后一步需要输出，那么它只在最后一步执行一次。

##### 6.3 所以完整模型其实是两层配合

整个前向传播，不应该只看成“RNN自己在工作”，而应该看成：

* RNN层负责时序建模
* 全连接层负责输出映射

> RNN层先提取时序特征，全连接层再把这些特征转换成最终任务输出。

#### 7. 隐藏状态在前向传播中到底起什么作用？ 🧠

隐藏状态是这一节最核心的主角。

##### 7.1 它在保存历史信息

如果没有隐藏状态，那么模型每次只能看当前输入 $x_t$

这样的话，每个时间步彼此独立，模型就退化成普通网络，无法利用前文信息。

而有了隐藏状态 $h_{t-1}$，RNN 就可以把前面时刻的信息带到当前时刻。

所以隐藏状态的第一个作用是：

> 记住前面的内容

##### 7.2 它在连接不同时间步

前向传播之所以能“串起来”，靠的正是：

$h_{t-1} \rightarrow h_t \rightarrow h_{t+1}$

所以隐藏状态的第二个作用是：

> 作为不同时间步之间的信息桥梁

##### 7.3 它在不断更新记忆

隐藏状态不是固定不变的，而是在每个时间步都重新计算。

也就是说：

* 第 $1$ 步得到 $h_1$
* 第 $2$ 步更新成 $h_2$
* 第 $3$ 步更新成 $h_3$

所以隐藏状态的第三个作用是：

> 不断融合新的输入和旧的记忆，形成新的记忆

#### 8. 前向传播中的参数会不会变化？ 🔁

这是一个必须明确的问题。

在前向传播的不同时间步中，下面这些参数都不变：

* $W_x$
* $W_h$
* $W_y$
* $b$
* $b_y$

变化的是：

* 当前输入 $x_t$
* 上一时刻隐藏状态 $h_{t-1}$
* 当前新得到的隐藏状态 $h_t$
* 当前输出 $y_t$

也就是说：

> RNN 的前向传播是在不同时间步重复使用同一套参数。

这正是参数共享的体现。

#### 9. 不同输出模式下，前向传播有什么区别？ 📚

虽然前向传播的核心更新公式都一样，但不同任务中，全连接层“什么时候接、接几次”，会不同。

##### 9.1 $N:1$ 模式

例如文本分类。

这种任务中，前向传播重点在于：

* RNN层先把整个输入序列都读完
* 不断更新隐藏状态
* 最后只取最终隐藏状态 $h_T$
* 再接一次全连接层得到结果

也就是说：

> 前面时间步主要负责积累信息，最后一步负责给结果。

##### 9.2 $N:N$ 模式

例如词性标注、命名实体识别。

这种任务中，前向传播重点在于：

* 每一步输入一个 $x_t$
* 每一步更新一个 $h_t$
* 每一步都接一次全连接层，输出一个 $y_t$

也就是说：

> 每一个时间步的隐藏状态都会被映射成输出。

##### 9.3 $1:N$ 模式

例如某些生成任务。

这种任务中，前向传播重点在于：

* 先给一个起始输入或初始状态
* 然后模型一步一步生成隐藏状态
* 每一步隐藏状态再经过全连接层生成当前输出

所以不同输出模式的区别，本质上不是 RNN层的公式变了，而是：

> 全连接层在时间维度上的使用方式变了。